In [1]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 72.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 33.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.1.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [22]:
import os
import torch
import pandas as pd
import numpy as np
import random

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

In [6]:
class Config:
    MOVIES_PATH = r'/kaggle/input/d/rounakbanik/the-movies-dataset/movies_metadata.csv'
    RATINGS_PATH = r'/kaggle/input/d/rounakbanik/the-movies-dataset/ratings.csv'
    
 
    MODEL_DIRS = [
        "/kaggle/input/gemma-2/transformers/gemma-2-9b-it/2",
        "/kaggle/input/gemma-2/transformers/gemma-2-9b-it/1"
    ]
    MODEL_ID = next((p for p in MODEL_DIRS if os.path.exists(p)), "google/gemma-2-9b-it")
    
    TOP_MOVIES = 1500  
    MIN_VOTES = 20     
    MAX_HIST = 15

In [7]:
def load_and_prep_data():
    
    cols = ['id', 'title', 'genres', 'overview', 'vote_count']
    movies = pd.read_csv(Config.MOVIES_PATH, low_memory=False, usecols=cols)
    movies = movies[pd.to_numeric(movies['id'], errors='coerce').notnull()]
    movies['id'] = movies['id'].astype(int)
    movies['vote_count'] = pd.to_numeric(movies['vote_count'], errors='coerce').fillna(0)
    
    top_movies = movies.sort_values('vote_count', ascending=False).head(Config.TOP_MOVIES).copy()
    
    def format_movie(row):
        try:
            genres = [x['name'] for x in eval(row['genres'])][:3]
            g_str = ", ".join(genres)
        except:
            g_str = "Unknown"
        return f"{row['title']} ({g_str})"

    top_movies['llm_text'] = top_movies.apply(format_movie, axis=1)
    
    ratings = pd.read_csv(Config.RATINGS_PATH)
    ratings = ratings[ratings['movieId'].isin(top_movies['id'])]
    ratings = ratings[ratings['rating'] >= 3.8] 
    
    user_history = ratings.groupby('userId')['movieId'].apply(list).reset_index()
    user_history = user_history[user_history['movieId'].apply(len) >= Config.MIN_VOTES]
    
    movie_map = top_movies.set_index('id')['llm_text'].to_dict()
    
    def get_history_str(m_ids):
        return "\n".join([f"- {movie_map[m]}" for m in m_ids[-Config.MAX_HIST:] if m in movie_map])
        
    user_history['history_str'] = user_history['movieId'].apply(get_history_str)
    
    return user_history

In [9]:
df_users = load_and_prep_data()

In [11]:
def load_llm():
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        Config.MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        local_files_only=os.path.exists(Config.MODEL_ID) 
    )
    
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=400,
        temperature=0.7,
        return_full_text=False
    )
    
    return pipe, tokenizer

In [12]:
llm_pipe, tokenizer = load_llm()

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [14]:
def query_llm(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipe(prompt)
    return out[0]['generated_text'].strip()

In [24]:


target = df_users.sample(1).iloc[0]
user_id = target['userId']
history = target['history_str']





msg_rec = [{"role": "user", "content": f"""
You are a Senior Film Curator AI. Your task is to analyze a user's viewing history and recommend high-quality hidden gems or masterpieces.

User's History:
{history}

INSTRUCTIONS:
Step 1: Analyze the "Psychological Profile" of the user. What emotions do they seek? (e.g., Adrenaline, Intellectual challenge, Escapism).
Step 2: Identify the specific sub-genres they love (e.g., not just "Sci-Fi", but "Cyberpunk" or "Space Opera").
Step 3: Recommend EXACTLY 5 NEW movies. 
   - DO NOT recommend movies already in the history.
   - Diversity is key: include at least one older classic and one modern hit.

OUTPUT FORMAT:
---
**User Profile:** [1-2 sentences deep analysis]
**Top 5 Recommendations:**
1. **[Movie Title]** (Year)
   * Why: [Connect it specifically to a movie they watched. E.g., "Since you liked X, this offers similar tension but..."]
   * Predicted Score: [X.X/10]

2. **[Movie Title]** (Year)
   * Why: [...]
   * Predicted Score: [X.X/10]

3. **[Movie Title]** (Year)
   * Why: [...]
   * Predicted Score: [X.X/10]

4. **[Movie Title]** (Year)
   * Why: [...]
   * Predicted Score: [X.X/10]

5. **[Movie Title]** (Year)
   * Why: [...]
   * Predicted Score: [X.X/10]
---
"""}]

print(query_llm(msg_rec))



candidates = df_users[df_users['userId'] != user_id].sample(1)

for _, cand in candidates.iterrows():
    cand_id = cand['userId']
    cand_hist = cand['history_str']
    
    print(f"\ncomparing target user {user_id} with candidate {cand_id}")
    print(f" candidate history:\n{cand_hist}")
    
    msg_sim = [{"role": "user", "content": f"""
    Act as a Complex Collaborative Filtering Algorithm.
    
    TARGET USER A:
    {history}
    
    CANDIDATE USER B:
    {cand_hist}
    
    LOGIC STEPS:
    1. Compare the lists. Look for subtle connections (e.g., both like Dark Comedies, even if movies are different).
    2. Calculate a "Compatibility Index" (0-100%).
       - <40%: Not compatible.
       - 40-70%: Casual friends.
       - >70%: Soulmates.
    3. IF Index > 50%: Select 5 movies from User B that User A MUST see.
    4. IF Index <= 50%: Reply only with "Not compatible enough."
    
    OUTPUT FORMAT:
    **Compatibility Index:** [X%]
    **Verdict:** [Short analysis of shared taste]
    
    **Recommendations (Only if compatible):**
    1. [Title] - [Why it fits User A]
    2. [Title] - [Why it fits User A]
    3. [Title] - [Why it fits User A]
    4. [Title] - [Why it fits User A]
    5. [Title] - [Why it fits User A]
    """}]
    
    res = query_llm(msg_sim)
    print(res)

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


---
**User Profile:** This user enjoys a mix of thrilling narratives with a leaning towards character-driven stories, whether it's the psychological turmoil in "The Talented Mr. Ripley" or the emotional journey in "Bridge to Terabithia." They appreciate well-crafted action sequences but also seem to enjoy films with a touch of humor and fantasy.

**Top 5 Recommendations:**
1. **The Silence of the Lambs (1991)**
   * Why:  Similar to "The Talented Mr. Ripley," this film delves into the mind of a captivating yet terrifying criminal.  Expect intense psychological suspense and remarkable performances.
   * Predicted Score: 9.2/10

2. **Moonrise Kingdom (2012)**
   * Why:  Combining the coming-of-age themes of "Bridge to Terabithia" with the whimsical touch of "Beetlejuice," this Wes Anderson film follows a pair of young runaways with a unique and enchanting story.
   * Predicted Score: 8.5/10

3. **Blade Runner 2049 (2017)**
   * Why: If the futuristic world of "I, Robot" and the action of